In [66]:
import random

In [67]:
@dataclass(frozen=True)
class Config:
    population_size: int = 100
    parent_selection_count: int = 5
    ga_pipeline_rounds: int = 1000
    max_evaluations: int = 10000
    n_queens: int = 8
    mutation_probability: float = 0.5
    crossover_probability: float = 1.0
    mutation_type: str = "swap"


cfg = Config()

In [ ]:

# Set of Utility Functions
def swap(item1, item2):
    return item2, item1

def select_a_random_chromosome(N=cfg.n_queens):
    return random.randint(0, N - 1)

def select_a_random_phenotype(population_size):
    return random.randint(0, population_size - 1)

def generate_chromosome(N=cfg.n_queens):
    return random.sample(range(N), N)

def ga_summary(
        original_population,
        parents,
        crossover_result,
        crossover_mode,
        mutated_children,
        surival_selection_type,
        mean_fitness, evaluations
    ):

    print("\n├──----[ GA Round Summary ]-----")
    print(f"├──> Evaluations: {evaluations}")
    print(f"├──> Original Population Size: {len(original_population)}")
    print(f"├── Selected Parents:")
    print("├──>", parents)
    print(f"├── Crossver Oprator: {crossover_mode}")
    print(f"├── Crossover Result:")
    print("├──>", crossover_result)
    print(f"├──> Survival Selection Type: {surival_selection_type}")
    print(f"├──> Mutated Children:")
    print("├──", mutated_children)
    print(f"└──[ Mean Fitness: {mean_fitness} ]----\n")

In [69]:
def generate_population(size, N=cfg.n_queens):
    return [generate_chromosome(N) for _ in range(size)]

In [70]:
def fitness_evaluation(queens):
    penalty = 0
    n = len(queens)
    for i in range(n):
        for j in range(i + 1, n):
            if queens[i] == queens[j]:
                penalty += 1
            elif abs(queens[i] - queens[j]) == abs(i - j):
                penalty += 1
    fitness = 1 / (1 + penalty)
    return round(fitness, 3)

In [71]:
def parent_selection(selection_count, population):
    sample = [population[select_a_random_phenotype(len(population) - 1)]
              for _ in range(selection_count)]
    parents = [{"fitness": fitness_evaluation(s), "chromosome": s} for s in sample]
    parents.sort(key=lambda x: x['fitness'], reverse=True)
    return parents[0], parents[1]

In [72]:
def crossover(parent1, parent2, prob=cfg.crossover_probability, mode="cutfill", cuts=1):
    if random.random() > prob:
        return [parent1[:], parent2[:]]

    N = len(parent1)

    #CUT-AND-FILL
    if mode == "cutfill":
        crossover_point = select_a_random_chromosome(N)
        if crossover_point < 1:
            crossover_point = 3

        p1_first = parent1[:crossover_point]
        p2_cycle = parent2[crossover_point:] + parent2[:crossover_point]
        child1_tail = [g for g in p2_cycle if g not in p1_first][: N - crossover_point]
        child1 = p1_first + child1_tail

        p2_first = parent2[:crossover_point]
        p1_cycle = parent1[crossover_point:] + parent1[:crossover_point]
        child2_tail = [g for g in p1_cycle if g not in p2_first][: N - crossover_point]
        child2 = p2_first + child2_tail
        return [child1, child2]

    #PMX CROSSOVER
    if mode == "pmx":
        c1, c2 = sorted(random.sample(range(N), 2))
        child1, child2 = parent1[:], parent2[:]

        child1[c1:c2], child2[c1:c2] = parent2[c1:c2], parent1[c1:c2]

        # mapping
        mapping1 = {parent2[i]: parent1[i] for i in range(c1, c2)}
        mapping2 = {parent1[i]: parent2[i] for i in range(c1, c2)}

        # fix duplicates
        def map_gene(gene, mapping):
            while gene in mapping:
                gene = mapping[gene]
            return gene

        for i in list(range(0, c1)) + list(range(c2, N)):
            child1[i] = map_gene(child1[i], mapping1)
            child2[i] = map_gene(child2[i], mapping2)
        return [child1, child2]

    #MULTI-CUT (2 or 3 cuts)
    if mode == "multi":
        cuts = min(cuts, 3)
        cut_points = sorted(random.sample(range(1, N - 1), cuts))
        parts1, parts2 = [], []
        last = 0
        for cp in cut_points + [N]:
            parts1.append(parent1[last:cp])
            parts2.append(parent2[last:cp])
            last = cp
        child1, child2 = [], []
        for i in range(len(parts1)):
            if i % 2 == 0:
                child1 += parts1[i]
                child2 += parts2[i]
            else:
                child1 += parts2[i]
                child2 += parts1[i]
        return [child1, child2]
    return [parent1[:], parent2[:]]

In [73]:
def mutation(chromosome, prob=cfg.mutation_probability, mode=cfg.mutation_type):
    if random.random() > prob:
        return chromosome

    N = len(chromosome)
    i, j = select_a_random_chromosome(N), select_a_random_chromosome(N)
    if mode == "swap":
        chromosome[i], chromosome[j] = chromosome[j], chromosome[i]
    elif mode == "bitwise":
        chromosome[i] = random.choice([x for x in range(N) if x not in chromosome or x == chromosome[i]])
    return chromosome

In [74]:
def survival_selection(population, children, population_size=cfg.population_size, elitism=False, elite_count=2):
    all_samples = population + children
    fitnesses = [{"fitness": fitness_evaluation(s), "chromosome": s} for s in all_samples]
    fitnesses.sort(key=lambda x: x['fitness'], reverse=True)

    if elitism:
        elites = fitnesses[:elite_count]
        remaining = fitnesses[elite_count:]
        next_gen = elites + remaining[: population_size - elite_count]
        return [f["chromosome"] for f in next_gen]
    else:
        return [f["chromosome"] for f in fitnesses[:population_size]]


In [75]:
def fitness_mean(population):
    return round(sum(fitness_evaluation(s) for s in population) / len(population), 4)

In [ ]:

def simple_GA_pipeline(
    crossover_mode="cutfill", 
    cuts=1, 
    elitism=False
):
    population = generate_population(cfg.population_size, cfg.n_queens)
    evaluations = 0

    for gen in range(cfg.ga_pipeline_rounds):
        parents = parent_selection(cfg.parent_selection_count, population)
        evaluations += len(parents)

        crossover_result = crossover(
            parents[0]['chromosome'], 
            parents[1]['chromosome'],
            prob=cfg.crossover_probability,
            mode=crossover_mode,
            cuts=cuts
        )
        evaluations += 2

        children = [mutation(c, cfg.mutation_probability, cfg.mutation_type) for c in crossover_result]
        evaluations += len(children)

        population = survival_selection(population, children, cfg.population_size, elitism=elitism)
        mean_fitness = fitness_mean(population)

        if any(fitness_evaluation(p) == 1 for p in population):
            print(f"✅ Solution found at generation {gen} after {evaluations} evaluations!")
            print("Solution:", [p for p in population if fitness_evaluation(p) == 1][0])
            break
        if evaluations >= cfg.max_evaluations:
            print("Terminated after reaching evaluation limit.")
            break

        ga_summary(
            original_population=population,
            parents=parents,
            surival_selection_type= elitism == True and "Elitism" or "Standard",
            crossover_mode= crossover_mode,
            crossover_result=crossover_result,
            mutated_children=children,
            mean_fitness=mean_fitness,
            evaluations=evaluations
        )
    return population

In [80]:
simple_GA_pipeline(crossover_mode="cutfill", elitism=False)


├──----[ GA Round Summary ]-----
├──> Evaluations: 6
├──> Original Population Size: 100
├── Selected Parents:
├──> ({'fitness': 0.25, 'chromosome': [4, 2, 1, 5, 0, 6, 3, 7]}, {'fitness': 0.167, 'chromosome': [3, 4, 1, 0, 6, 5, 7, 2]})
├── Crossver Oprator: cutfill
├── Crossover Result:
├──> [[4, 2, 3, 0, 6, 5, 7, 1], [3, 4, 1, 5, 0, 6, 7, 2]]
├──> Survival Selection Type: Standard
├──> Mutated Children:
├── [[4, 2, 3, 0, 6, 5, 7, 1], [3, 4, 1, 5, 0, 6, 7, 2]]
└──[ Mean Fitness: 0.1937 ]----


├──----[ GA Round Summary ]-----
├──> Evaluations: 12
├──> Original Population Size: 100
├── Selected Parents:
├──> ({'fitness': 0.333, 'chromosome': [3, 0, 4, 7, 1, 2, 6, 5]}, {'fitness': 0.25, 'chromosome': [6, 0, 3, 2, 5, 7, 4, 1]})
├── Crossver Oprator: cutfill
├── Crossover Result:
├──> [[3, 0, 4, 2, 5, 7, 1, 6], [6, 1, 3, 7, 0, 2, 5, 4]]
├──> Survival Selection Type: Standard
├──> Mutated Children:
├── [[3, 0, 4, 2, 5, 7, 1, 6], [6, 1, 3, 7, 0, 2, 5, 4]]
└──[ Mean Fitness: 0.1975 ]----


├─

[[5, 2, 6, 1, 7, 4, 0, 3],
 [4, 1, 3, 0, 5, 7, 2, 6],
 [4, 6, 0, 2, 5, 1, 3, 7],
 [3, 1, 6, 2, 0, 4, 7, 5],
 [5, 1, 4, 7, 2, 6, 3, 0],
 [4, 6, 0, 2, 5, 1, 3, 7],
 [6, 0, 3, 5, 7, 2, 4, 1],
 [6, 0, 3, 5, 7, 1, 4, 2],
 [3, 0, 4, 7, 1, 2, 6, 5],
 [7, 1, 5, 3, 6, 0, 2, 4],
 [0, 7, 3, 5, 2, 4, 1, 6],
 [1, 4, 6, 7, 0, 3, 5, 2],
 [7, 4, 2, 5, 0, 1, 3, 6],
 [7, 4, 6, 0, 2, 5, 1, 3],
 [6, 1, 3, 7, 0, 2, 5, 4],
 [6, 1, 3, 0, 7, 5, 4, 2],
 [6, 1, 3, 7, 0, 2, 4, 5],
 [4, 5, 1, 6, 2, 0, 7, 3],
 [4, 7, 5, 2, 0, 6, 3, 1],
 [2, 5, 6, 1, 4, 0, 7, 3],
 [0, 6, 4, 7, 5, 3, 1, 2],
 [3, 1, 6, 2, 7, 5, 0, 4],
 [0, 7, 3, 4, 2, 5, 1, 6],
 [2, 5, 1, 6, 7, 3, 0, 4],
 [0, 5, 7, 4, 3, 6, 2, 1],
 [2, 4, 5, 7, 1, 6, 0, 3],
 [6, 1, 3, 0, 7, 5, 2, 4],
 [5, 0, 7, 4, 2, 6, 3, 1],
 [5, 2, 6, 1, 4, 0, 7, 3],
 [6, 5, 7, 2, 1, 3, 4, 0],
 [4, 2, 1, 5, 0, 6, 3, 7],
 [5, 1, 3, 7, 2, 4, 0, 6],
 [5, 2, 7, 1, 4, 6, 3, 0],
 [4, 1, 2, 5, 7, 6, 0, 3],
 [4, 3, 6, 2, 1, 5, 7, 0],
 [2, 7, 3, 4, 0, 5, 6, 1],
 [4, 7, 5, 2, 0, 6, 1, 3],
 

In [78]:
simple_GA_pipeline(crossover_mode="multi", cuts=3)


├──----[ GA Round Summary ]-----
├──> Evaluations: 6
├──> Original Population Size: 100
├── Selected Parents:
├──> ({'fitness': 0.2, 'chromosome': [7, 4, 1, 5, 2, 3, 0, 6]}, {'fitness': 0.2, 'chromosome': [3, 2, 6, 1, 4, 5, 7, 0]})
├── Crossver Oprator: multi
├── Crossover Result:
├──> [[7, 2, 6, 5, 2, 3, 7, 0], [0, 4, 1, 1, 4, 5, 3, 6]]
├──> Survival Selection Type: Standard
├──> Mutated Children:
├── [[7, 2, 6, 5, 2, 3, 7, 0], [0, 4, 1, 1, 4, 5, 3, 6]]
└──[ Mean Fitness: 0.201 ]----


├──----[ GA Round Summary ]-----
├──> Evaluations: 12
├──> Original Population Size: 100
├── Selected Parents:
├──> ({'fitness': 0.25, 'chromosome': [4, 2, 3, 6, 0, 7, 1, 5]}, {'fitness': 0.25, 'chromosome': [7, 3, 5, 0, 4, 1, 2, 6]})
├── Crossver Oprator: multi
├── Crossover Result:
├──> [[4, 2, 5, 6, 4, 1, 2, 6], [7, 0, 3, 0, 3, 7, 1, 5]]
├──> Survival Selection Type: Standard
├──> Mutated Children:
├── [[4, 2, 5, 6, 4, 1, 2, 6], [7, 0, 3, 0, 3, 7, 1, 5]]
└──[ Mean Fitness: 0.202 ]----


├──----[ GA 

[[1, 4, 6, 3, 0, 7, 5, 2],
 [4, 2, 5, 7, 1, 3, 0, 6],
 [6, 1, 3, 5, 0, 7, 4, 2],
 [1, 6, 2, 7, 4, 0, 3, 5],
 [1, 4, 0, 5, 7, 2, 6, 3],
 [2, 3, 7, 4, 1, 5, 0, 6],
 [2, 0, 5, 7, 4, 1, 3, 5],
 [2, 0, 3, 1, 7, 5, 6, 4],
 [1, 7, 4, 0, 6, 3, 5, 2],
 [6, 3, 1, 4, 2, 0, 5, 7],
 [5, 0, 2, 6, 4, 1, 7, 3],
 [1, 4, 6, 7, 0, 3, 5, 2],
 [3, 0, 5, 1, 6, 4, 2, 7],
 [5, 3, 4, 0, 7, 1, 6, 2],
 [0, 5, 4, 1, 3, 6, 7, 2],
 [1, 4, 6, 0, 6, 3, 5, 2],
 [4, 7, 5, 6, 1, 2, 0, 3],
 [6, 3, 4, 2, 7, 5, 1, 0],
 [4, 2, 3, 6, 0, 7, 1, 5],
 [7, 2, 4, 1, 0, 3, 5, 6],
 [5, 7, 2, 1, 4, 6, 0, 3],
 [2, 3, 7, 6, 1, 5, 0, 4],
 [5, 4, 1, 6, 0, 3, 7, 2],
 [6, 1, 4, 0, 5, 3, 2, 7],
 [2, 6, 7, 3, 1, 4, 0, 5],
 [3, 7, 4, 0, 5, 1, 6, 2],
 [5, 2, 1, 7, 4, 3, 6, 0],
 [2, 5, 4, 6, 1, 0, 7, 3],
 [6, 1, 2, 0, 5, 4, 7, 3],
 [7, 3, 5, 0, 4, 1, 2, 6],
 [3, 1, 7, 6, 4, 2, 0, 5],
 [7, 0, 6, 4, 1, 5, 3, 2],
 [5, 3, 0, 2, 6, 4, 7, 1],
 [7, 3, 4, 6, 0, 2, 5, 1],
 [7, 2, 0, 7, 5, 3, 6, 4],
 [5, 4, 1, 6, 0, 3, 7, 2],
 [5, 2, 1, 7, 4, 3, 6, 0],
 

In [82]:
import matplotlib.pyplot as plt

def plot_convergence(fitness_history, label):
    plt.plot(fitness_history, label=label)
    plt.xlabel("Generation")
    plt.ylabel("Mean Fitness")
    plt.title("GA Convergence Curve")
    plt.legend()